# 📓 Retrieval Baseline Evaluierung(Docling-Collection)

**Autor:** Sakina Ahmadi
**Beschreibung:** Dieses Notebook führt eine automatisierte Retrieval-Evaluierung
der Docling-Collection (`stage2_docling_hybrid_bge`) durch.

## Ziel
Wir evaluieren die Retrieval-Qualität unserer RAG-Pipeline anhand von 84
Gold-Standard-Fragen und berechnen:
- **Hit Rate @5 und @10**
- **Mean Reciprocal Rank (MRR)**
- **Durchschnittliche Latenz pro Query**

## Pipeline
1. Frage mit BGE-M3 embedden
2. Suche in Qdrant (Cosine Similarity, HNSW-Index)
3. Ergebnisse mit Gold-Standard vergleichen
4. Metriken berechnen und speichern

---
## 1. Installation & Importe

Wir installieren die benötigten Pakete und importieren alle Bibliotheken.

In [6]:
# =====================================================================
# 1. DEPENDENCIES & SETUP
# =====================================================================
!pip install sentence-transformers qdrant-client tqdm

import json
import os
import re
import time
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from kaggle_secrets import UserSecretsClient

🖥️ Initialisiere Embedding-Modell BAAI/bge-m3 auf GPU...


---
## 2. Konfiguration

Hier definieren wir alle Parameter für die Evaluierung:
- **Collection:** `stage2_docling_hybrid_bge` (Docling, Hybrid Chunking, INT8)
- **Embedding:** BAAI/bge-m3 (1024 Dimensionen)
- **HNSW:** ef_search = 64
- **Top-K:** 5 und 10

In [6]:
# =====================================================================
# 2. KONFIGURATION
# =====================================================================
EVAL_SET_PATH = "/kaggle/input/datasets/sakinaahmadi/rag-automl-gold-standard-100q/eval_set_100q.jsonl"
COLLECTION_NAME = "stage2_docling_hybrid_bge"
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda"

# AutoML-Stellschrauben für den ersten Test
TOP_K_VALUES = [5, 10]
EF_SEARCH = 64

---
## 3. Hilfsfunktion: ID-Normalisierung

Diese Funktion normalisiert Paper-IDs, indem sie Versionsnummern (v1, v2, ...)
entfernt und Punkte in Slashes umwandelt.

**Beispiel:** `2512.02339v1` → `2512/02339`

In [6]:
# =====================================================================
# 3. HILFSFUNKTION
# =====================================================================
def normalize_paper_id(pid: str) -> str:
    """Normalisiert Paper-IDs: entfernt Versionen, wandelt Punkte in Slashes."""
    pid = re.sub(r'v\d+$', '', pid)  # Version hinten abschneiden
    pid = pid.replace(".", "/")  # Punkte in Slashes
    return pid

---
## 4. Hauptfunktion: Retrieval-Evaluierung

Die Hauptfunktion führt folgende Schritte aus:
1. Embedding-Modell laden (BGE-M3 auf GPU)
2. Verbindung zu Qdrant Cloud herstellen
3. Gold-Standard-Fragen laden (84 Fragen)
4. Für jede Frage: Embedding → Suche → Vergleich
5. Metriken berechnen: Hit Rate, MRR, Latenz
6. Ergebnisse speichern

In [6]:
# =====================================================================
# 4. HAUPTFUNKTION
# =====================================================================
def main():
    print(f"🖥️ Initialisiere Embedding-Modell {EMBEDDING_MODEL_NAME} auf GPU...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
    
    print("🌐 Verbinde mit Qdrant Cloud...")
    user_secrets = UserSecretsClient()
    client = QdrantClient(
        url=user_secrets.get_secret("QDRANT_URL"),
        api_key=user_secrets.get_secret("QDRANT_API_KEY"),
        check_compatibility=False
    )
    
    # Gold Standard laden
    if not os.path.exists(EVAL_SET_PATH):
        print(f"❌ FEHLER: Gold Standard Datei nicht gefunden unter {EVAL_SET_PATH}")
        return
    
    queries = []
    with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    
    print(f"📖 {len(queries)} Testfragen erfolgreich aus dem Gold Standard geladen.")

🌐 Verbinde mit Qdrant Cloud...
📖 84 Testfragen erfolgreich aus dem Gold Standard geladen.
🚀 Starte Retrieval-Evaluierung...


---
## 5. Evaluierungsschleife

Für jede der 84 Fragen:
1. Frage in Vektor umwandeln (BGE-M3)
2. In Qdrant suchen (top_k=10, hnsw_ef=64)
3. Gefundene Paper-IDs mit der erwarteten ID vergleichen
4. Hit Rate und MRR berechnen

In [6]:
    # Zähler für die Metriken initialisieren
    hits_at_k = {k: 0 for k in TOP_K_VALUES}
    rr_scores = []
    search_times = []
    
    print("\n🚀 Starte Retrieval-Evaluierung...")
    
    # Schleife über alle Fragen
    for q_data in tqdm(queries, desc="Evaluating Queries"):
        question = q_data["query"]
        
        # 1. Frage in Vektor umwandeln
        start_time = time.time()
        query_vector = model.encode(question, show_progress_bar=False).tolist()
        
        # 2. In Qdrant suchen
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=max(TOP_K_VALUES),
            search_params={"hnsw_ef": EF_SEARCH},
            with_payload=["paper_id"]
        ).points
        
        search_times.append(time.time() - start_time)
        
        # 3. Auswertung
        target_paper_id = normalize_paper_id(q_data["expected_paper"])
        
        retrieved_ids = []
        for hit in search_result:
            if hit.payload and "paper_id" in hit.payload:
                retrieved_ids.append(normalize_paper_id(hit.payload["paper_id"]))
        
        # Hit Rate berechnen
        for k in TOP_K_VALUES:
            if target_paper_id in retrieved_ids[:k]:
                hits_at_k[k] += 1
        
        # Reciprocal Rank berechnen
        if target_paper_id in retrieved_ids:
            rank = retrieved_ids.index(target_paper_id) + 1
            rr_scores.append(1.0 / rank)
        else:
            rr_scores.append(0.0)

Evaluating Queries: 100%|██████████| 84/84 [00:13<00:00, 6.04it/s]


---
## 6. Ergebnisse ausgeben

Nach der Evaluierung geben wir die Metriken aus und speichern sie als JSON.

In [6]:
    # Metriken finalisieren
    total_queries = len(queries)
    hit_rate_at_5 = hits_at_k[5] / total_queries
    hit_rate_at_10 = hits_at_k[10] / total_queries
    mrr = sum(rr_scores) / total_queries
    avg_search_time_ms = (sum(search_times) / total_queries) * 1000

    # Ausgabe
    print("\n" + "="*60)
    print("📊 ERGEBNISSE DEINER RETRIEVAL BASELINE (STAGE 2)")
    print("="*60)
    print(f"🔹 Getestete Fragen:      {total_queries}")
    print(f"🔹 Genutztes Modell:      {EMBEDDING_MODEL_NAME}")
    print(f"🔹 Such-Parameter:        hnsw_ef = {EF_SEARCH}")
    print("-" * 60)
    print(f"🎯 Hit Rate @ 5:          {hit_rate_at_5 * 100:.2f} %")
    print(f"🎯 Hit Rate @ 10:         {hit_rate_at_10 * 100:.2f} %")
    print(f"📈 Mean Reciprocal Rank:  {mrr:.4f}")
    print(f"⚡ Avg Latency pro Query: {avg_search_time_ms:.2f} ms")
    print("="*60)
    
    # Ergebnisse speichern
    os.makedirs("/kaggle/working", exist_ok=True)
    baseline_results = {
        "model": EMBEDDING_MODEL_NAME, "hnsw_ef": EF_SEARCH,
        "hit_rate_at_5": hit_rate_at_5, "hit_rate_at_10": hit_rate_at_10,
        "mrr": mrr, "avg_latency_ms": avg_search_time_ms
    }
    with open("/kaggle/working/retrieval_baseline_results.json", "w") as json_f:
        json.dump(baseline_results, json_f, indent=4)
    print("💾 Metriken wurden unter '/kaggle/working/retrieval_baseline_results.json' gesichert.")

if __name__ == "__main__":
    main()


📊 ERGEBNISSE DEINER RETRIEVAL BASELINE (STAGE 2)
🔹 Getestete Fragen:      84
🔹 Genutztes Modell:      BAAI/bge-m3
🔹 Such-Parameter:        hnsw_ef = 64
------------------------------------------------------------
🎯 Hit Rate @ 5:          83.33 %
🎯 Hit Rate @ 10:         86.90 %
📈 Mean Reciprocal Rank:  0.7827
⚡ Avg Latency pro Query: 164.59 ms
💾 Metriken wurden unter '/kaggle/working/retrieval_baseline_results.json' gesichert.


---
## 7. Zusammenfassung der Ergebnisse

Die Retrieval-Baseline zeigt eine solide Performance:

| Metrik | Wert |
|--------|------|
| **Hit Rate @ 5** | **83,33 %** |
| **Hit Rate @ 10** | **86,90 %** |
| **MRR** | **0,7827** |
| **Latenz** | **164,59 ms** |

### Interpretation
- Die Hit Rate @ 10 von 86,90 % bedeutet, dass in fast 9 von 10 Fällen
  das richtige Paper in den ersten 10 Ergebnissen gefunden wird.
- Der MRR von 0,7827 zeigt, dass das richtige Paper im Durchschnitt
  auf Platz 1,28 gefunden wird.
- Die Latenz von 164,59 ms pro Query ist für eine Echtzeitanwendung akzeptabel.

Diese Werte dienen als Baseline für die weiteren AutoML-Experimente.